# RAG 的测试集生成

这个简单的指南将帮助您使用自己的文档生成用于评估 RAG 管道的测试集。

## 快速入门
让我们快速看一个为 RAG 流水线生成测试集的示例。接下来，我们将探讨测试集生成流水线的主要组件。

## 加载示例文档
为了本教程的目的，我们将使用此存储库中的示例文档。您可以将其替换为您自己的文档。

```shell
git clone https://huggingface.co/datasets/explodinggradients/Sample_Docs_Markdown
```

## 加载文档
现在，我们将使用 DirectoryLoader 从示例数据集中加载文档，它是 langchain_community 的文档加载器之一。您也可以使用`llama_index`中的任何加载器。

```shell
pip install langchain-community
```




In [ ]:
from langchain_community.document_loaders import DirectoryLoader

path = "Sample_Docs_Markdown/"
loader = DirectoryLoader(path, glob="**/*.md")
docs = loader.load()

## 选择你的LLM
您可以选择使用您选择的LLM

```shell
pip install langchain-openai
```

然后确保您的 OpenAI 密钥已准备好并在您的环境中可用

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your-openai-key"

将 LLM 包装起来LangchainLLMWrapper以便可以与 ragas 一起使用。

In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

## 生成测试集
现在我们将使用已加载的文档和 LLM 设置运行测试生成。如果您已使用llama_index加载文档的方式，请使用`generate_with_llama_index_docs`方法。

In [ ]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

## 分析测试集
生成测试集后，您需要查看它并选择您认为适合包含在最终测试集中的查询。您可以将测试集导出到 Pandas DataFrame 并对其进行各种分析。

In [ ]:
dataset.to_pandas()

## 深入了解
现在我们已经了解了如何生成测试集，让我们仔细看看测试集生成管道的主要组件以及如何快速自定义它。

核心是执行 2 个主要操作来生成测试集。

- 知识图谱创建：我们首先使用您提供的文档创建一个知识图谱，然后使用各种转换来丰富知识图谱，添加可用于生成测试集的附加信息。您可以从核心概念部分了解更多信息。
- 测试集生成：我们使用知识图谱 (KnowledgeGraph)生成一组场景。这些场景用于生成测试集。您可以从核心概念部分了解更多信息。


### 知识图谱创建
让我们首先使用之前加载的文档创建一个知识图谱。


In [2]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()

然后将文档添加到知识图谱中。


In [3]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )

NameError: name 'docs' is not defined

现在，我们将使用“转换”功能，通过附加信息来丰富知识图谱。在这里，我们将使用default_transforms创建一组默认转换，以便应用于您选择的 LLM 和嵌入模型。但您可以根据需要混合搭配这些转换，或构建您自己的转换。

In [ ]:
from ragas.testset.transforms import default_transforms, apply_transforms


# define your LLM and Embedding Model
# here we are using the same LLM and Embedding Model that we used to generate the testset
transformer_llm = generator_llm
embedding_model = generator_embeddings

trans = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, trans)

现在我们有了一个包含附加信息的知识图谱。您也可以保存它。

In [ ]:
kg.save("knowledge_graph.json")
loaded_kg = KnowledgeGraph.load("knowledge_graph.json")
loaded_kg

## 测试集生成
现在我们将使用它loaded_kg来创建TestsetGenerator。

In [ ]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loaded_kg)

我们还可以定义想要生成的查询的分布。这里我们使用默认分布。

In [ ]:
from ragas.testset.synthesizers import default_query_distribution

query_distribution = default_query_distribution(generator_llm)

现在我们可以生成测试集。

In [ ]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()